# MAMMAL Inference with the Fold Models

Score molecules with **all the fold models** from
[MAMMAL_finetuning_cv.ipynb](MAMMAL_finetuning_cv.ipynb) and average their scores (an ensemble). For a single
fine-tuned model, use [MAMMAL_inference.ipynb](MAMMAL_inference.ipynb).

An ensemble is usually a little more accurate than one model, but it is K times slower and the models are
loaded one at a time.

**Environment:** run with the `biomed-mammal` kernel.

**Input:** a `.csv` or `.parquet` file with a SMILES column. A 0/1 label column is optional; if present,
metrics are reported.

**Output**, in `<CV run folder>/predictions/<PREDICTION_NAME>/`:
```
predictions.csv   # every input row, plus the per-fold scores, their mean, pred and rank (1 = highest)
metrics.json      # only if the file has labels
config.json       # settings used
```

## 1. Inputs and settings

In [ ]:
from pathlib import Path

# Dataset name: all outputs go under aircheck/output/<DATASET>/
DATASET = "sample-data"

# Name of this prediction; outputs go to <CV run folder>/predictions/<PREDICTION_NAME>/
PREDICTION_NAME = "test"

# The cross-validation run folder from MAMMAL_finetuning_cv.ipynb (it holds fold_0/, fold_1/, ...)
CV_RUN_DIR = Path("../output") / DATASET / "mammal-cv"

# Molecules to score
DATA_PATH = Path("../../../biomed-multi-view/aircheck/sample_data/test.parquet")
SMILES_COL = "SMILES"
LABEL_COL = "LABEL"   # set to None if the file has no labels
KEEP_COLS = []        # other input columns to copy into predictions.csv, e.g. ["ID"]

In [ ]:
import os

BATCH_SIZE = 32
NUM_WORKERS = min(24, os.cpu_count())
THRESHOLD = 0.5           # score above this -> pred = 1
HIT_KS = [200, 100, 50]   # hit@K = number of label-1 molecules among the K highest-scored

# Prompt lengths, in tokens; keep these the same as the fine-tuning run
MAX_SMILES_TOKENS = 300
MAX_PROMPT_TOKENS = 320

In [ ]:
import json

import torch


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "mammal").is_dir() and (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError(f"Run this notebook from inside the biomed-multi-alignment repo (cwd={start})")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
CV_RUN_DIR = Path(CV_RUN_DIR).expanduser().resolve()
assert CV_RUN_DIR.is_dir(), f"CV run folder not found: {CV_RUN_DIR}"
assert Path(DATA_PATH).exists(), f"Data not found: {DATA_PATH}"

MODEL_PATHS = sorted((CV_RUN_DIR).glob("fold_*/best_model"), key=lambda p: int(p.parent.name.split("_")[1]))
assert MODEL_PATHS, f"No fold_*/best_model folders in {CV_RUN_DIR}"
N_FOLDS = len(MODEL_PATHS)
TOKENIZER_SOURCE = str(MODEL_PATHS[0])

OUT_DIR = CV_RUN_DIR / "predictions" / PREDICTION_NAME
assert not (OUT_DIR.exists() and any(OUT_DIR.iterdir())), (
    f"{OUT_DIR} already exists. Choose a new PREDICTION_NAME or delete that folder.")
OUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

config = dict(cv_run_dir=str(CV_RUN_DIR), models=[str(p) for p in MODEL_PATHS], n_folds=N_FOLDS,
              data_path=str(DATA_PATH), smiles_col=SMILES_COL, label_col=LABEL_COL,
              threshold=THRESHOLD, hit_ks=HIT_KS)
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=2))

print(f"Fold models : {N_FOLDS}")
print(f"Data        : {DATA_PATH}")
print(f"Output      : {OUT_DIR}")
print(f"Device      : {DEVICE}")

## 2. Prepare data

Molecules RDKit cannot read are kept in the output with an empty score, so every input row is accounted for.

In [ ]:
import numpy as np
import pandas as pd
import torch
from functools import partial
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from rdkit import Chem, RDLogger
from sklearn.metrics import average_precision_score, roc_auc_score

from fuse.data.utils.collates import CollateDefault
from mammal.keys import *
import pyarrow.parquet as pq
def read_table(path, columns, optional=()) -> pd.DataFrame:
    # Read only the needed columns from a .csv or .parquet file; optional columns are read if present
    path = Path(path)
    is_parquet = path.suffix.lower() in (".parquet", ".pq")
    available = pq.ParquetFile(path).schema_arrow.names if is_parquet else pd.read_csv(path, nrows=0).columns.tolist()
    missing = [c for c in columns if c not in available]
    assert not missing, f"{path.name} has no column(s) {missing}; available: {available}"
    wanted = list(dict.fromkeys([*columns, *(c for c in optional if c in available)]))
    df = pd.read_parquet(path, columns=wanted) if is_parquet else pd.read_csv(path, usecols=wanted)
    return df[wanted].reset_index(drop=True)
RDLogger.DisableLog("rdApp.*")

input_df = read_table(DATA_PATH, [SMILES_COL, *KEEP_COLS], optional=[LABEL_COL] if LABEL_COL else [])
input_df.insert(0, "row_index", range(len(input_df)))  # row in the input file
HAS_LABELS = LABEL_COL is not None and LABEL_COL in input_df.columns and input_df[LABEL_COL].notna().any()

smiles = input_df[SMILES_COL].astype(str).str.strip()
valid = input_df[SMILES_COL].notna() & smiles.map(lambda s: Chem.MolFromSmiles(s) is not None)
to_score = smiles[valid].tolist()

print(f"Rows           : {len(input_df):,}")
print(f"Valid SMILES   : {int(valid.sum()):,}")
if (~valid).any():
    print(f"Invalid SMILES : {int((~valid).sum()):,} (kept with an empty score), "
          f"e.g. {input_df.loc[~valid, SMILES_COL].head(3).tolist()}")
print(f"Labels         : {'yes' if HAS_LABELS else 'no (metrics will be skipped)'}")

## 3. Load the tokenizer and build the prompts

In [ ]:
import os
import warnings

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore", category=UserWarning, module="google.protobuf")

from fuse.data.tokenizers.modular_tokenizer.op import ModularTokenizerOp
from mammal.model import Mammal

tokenizer_op = ModularTokenizerOp.from_pretrained(TOKENIZER_SOURCE)

NEG_TOKEN_ID = tokenizer_op.get_token_id("<0>")
POS_TOKEN_ID = tokenizer_op.get_token_id("<1>")
PAD_TOKEN_ID = tokenizer_op.get_token_id("<PAD>")
print(f"Tokenizer loaded. Answer tokens: <0> = {NEG_TOKEN_ID}, <1> = {POS_TOKEN_ID}")

In [ ]:
IGNORE_INDEX = -100
TOKEN_WEIGHTS_KEY = "data.token_weights"
MAX_ANSWER_TOKENS = 4
ANSWER_POSITION = 1  # the decoder writes <SENTINEL_ID_0> first, then the answer


def encoder_prompt(smiles: str) -> str:
    """The text MAMMAL reads for one molecule."""
    return (
        f"<@TOKENIZER-TYPE=SMILES><SENTINEL_ID_0>"
        f"<MOLECULAR_ENTITY><MOLECULAR_ENTITY_SMALL_MOLECULE>"
        f"<@TOKENIZER-TYPE=SMILES@MAX-LEN={MAX_SMILES_TOKENS}>"
        f"<SEQUENCE_NATURAL_START>{smiles}<SEQUENCE_NATURAL_END><EOS>"
    )


def prepare_sample(sample_dict: dict, tokenizer_op, pos_weight: float = 1.0, with_label: bool = True) -> dict:
    """Tokenize one sample: the prompt, and (for training) the answer and its loss weight."""
    sample_dict[ENCODER_INPUTS_STR] = encoder_prompt(sample_dict["data.smiles"])
    tokenizer_op(
        sample_dict=sample_dict,
        key_in=ENCODER_INPUTS_STR,
        key_out_tokens_ids=ENCODER_INPUTS_TOKENS,
        key_out_attention_mask=ENCODER_INPUTS_ATTENTION_MASK,
        max_seq_len=MAX_PROMPT_TOKENS,
    )
    for k in (ENCODER_INPUTS_TOKENS, ENCODER_INPUTS_ATTENTION_MASK):
        sample_dict[k] = torch.tensor(sample_dict[k])

    if not with_label:
        return sample_dict

    label = int(sample_dict["data.label"])

    # What the model should write
    sample_dict[LABELS_STR] = f"<@TOKENIZER-TYPE=SMILES><SENTINEL_ID_0><{label}><EOS>"
    tokenizer_op(
        sample_dict=sample_dict, key_in=LABELS_STR,
        key_out_tokens_ids=LABELS_TOKENS, key_out_attention_mask=LABELS_ATTENTION_MASK,
        max_seq_len=MAX_ANSWER_TOKENS,
    )
    for k in (LABELS_TOKENS, LABELS_ATTENTION_MASK):
        sample_dict[k] = torch.tensor(sample_dict[k])
    # Padding must not count towards the loss
    sample_dict[LABELS_TOKENS][sample_dict[LABELS_TOKENS] == PAD_TOKEN_ID] = IGNORE_INDEX

    # What the decoder is fed during training (teacher forcing)
    sample_dict[DECODER_INPUTS_STR] = f"<@TOKENIZER-TYPE=SMILES><DECODER_START><SENTINEL_ID_0><{label}><EOS>"
    tokenizer_op(
        sample_dict=sample_dict, key_in=DECODER_INPUTS_STR,
        key_out_tokens_ids=DECODER_INPUTS_TOKENS, key_out_attention_mask=DECODER_INPUTS_ATTENTION_MASK,
        max_seq_len=MAX_ANSWER_TOKENS,
    )
    for k in (DECODER_INPUTS_TOKENS, DECODER_INPUTS_ATTENTION_MASK):
        sample_dict[k] = torch.tensor(sample_dict[k])

    # Class balancing: weight every counted position of this sample, 0 where the loss is ignored
    weight = pos_weight if label == 1 else 1.0
    sample_dict[TOKEN_WEIGHTS_KEY] = weight * (sample_dict[LABELS_TOKENS] != IGNORE_INDEX).float()
    return sample_dict


class MoleculeDataset(Dataset):
    """One row per molecule, from a DataFrame with 'smiles' and (for training) 'label' columns."""

    def __init__(self, df, tokenizer_op, pos_weight: float = 1.0, with_label: bool = True):
        self.smiles = df["smiles"].tolist()
        self.labels = df["label"].tolist() if with_label else [0] * len(df)
        self.tokenizer_op, self.pos_weight, self.with_label = tokenizer_op, pos_weight, with_label

    def __len__(self) -> int:
        return len(self.smiles)

    def __getitem__(self, idx: int) -> dict:
        sample = {"data.sample_id": idx, "data.smiles": self.smiles[idx], "data.label": self.labels[idx]}
        return prepare_sample(sample, self.tokenizer_op, self.pos_weight, self.with_label)


def make_collate():
    """Trim the padding each batch does not need (10-20% faster)."""
    return CollateDefault(special_handlers_keys={
        ENCODER_INPUTS_TOKENS: partial(CollateDefault.crop_padding, pad_token_id=PAD_TOKEN_ID),
        ENCODER_INPUTS_ATTENTION_MASK: partial(CollateDefault.crop_padding, pad_token_id=False),
    })
def compute_metrics(labels, scores, hit_ks=None, threshold=0.5) -> dict:
    """hit@K, AUROC, PR AUC and the threshold metrics, as the MMELON notebooks report them."""
    hit_ks = HIT_KS if hit_ks is None else hit_ks
    labels, scores = np.asarray(labels).astype(int), np.asarray(scores, dtype=float)
    top = np.argsort(-scores, kind="stable")
    metrics = {f"hit@{k}": int(labels[top[:k]].sum()) for k in hit_ks}
    pred = scores > threshold
    tp, fp = int((pred & (labels == 1)).sum()), int((pred & (labels == 0)).sum())
    tn, fn = int((~pred & (labels == 0)).sum()), int((~pred & (labels == 1)).sum())
    both_classes = len(set(labels.tolist())) == 2
    metrics.update(
        rocauc=float(roc_auc_score(labels, scores)) if both_classes else float("nan"),
        pr_auc=float(average_precision_score(labels, scores)) if both_classes else float("nan"),
        precision=tp / (tp + fp) if tp + fp else 0.0,
        recall=tp / (tp + fn) if tp + fn else 0.0,
        specificity=tn / (tn + fp) if tn + fp else 0.0,
        accuracy=(tp + tn) / len(labels) if len(labels) else float("nan"),
    )
    return metrics
@torch.no_grad()
def score_molecules(model, smiles_list, tokenizer_op, batch_size=BATCH_SIZE, device="cpu") -> np.ndarray:
    """The probability of `<1>` for each molecule, in the order given."""
    model = model.to(device).eval()
    dataset = MoleculeDataset(pd.DataFrame({"smiles": smiles_list}), tokenizer_op, with_label=False)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                        collate_fn=make_collate(), num_workers=NUM_WORKERS)
    out = []
    for batch in tqdm(loader, desc="Scoring"):
        for k in (ENCODER_INPUTS_TOKENS, ENCODER_INPUTS_ATTENTION_MASK):
            batch[k] = batch[k].to(device)
        result = model.generate(batch, output_scores=True, return_dict_in_generate=True, max_new_tokens=5)
        # SCORES is (batch, generated tokens, vocabulary); the model writes <SENTINEL_ID_0> then the answer
        probs = result[SCORES][:, ANSWER_POSITION, :].float().cpu()
        pos, neg = probs[:, POS_TOKEN_ID], probs[:, NEG_TOKEN_ID]
        out.append((pos / (pos + neg + 1e-8)).numpy())
    return np.concatenate(out)

## 4. Score the molecules

In [ ]:
%%time
import gc

fold_scores = {}
for i, path in enumerate(MODEL_PATHS):
    print(f"Fold {i + 1} of {N_FOLDS}: {path}")
    model = Mammal.from_pretrained(str(path))
    fold_scores[f"score_fold{i}"] = score_molecules(model, to_score, tokenizer_op,
                                                    batch_size=BATCH_SIZE, device=DEVICE)
    del model
    gc.collect()
    torch.cuda.empty_cache() if DEVICE == "cuda" else None

mean_scores = np.mean([fold_scores[k] for k in fold_scores], axis=0)

predictions = input_df.copy()
for name, values in fold_scores.items():
    predictions[name] = np.nan
    predictions.loc[valid, name] = values
predictions["score"] = np.nan
predictions.loc[valid, "score"] = mean_scores
predictions["pred"] = (predictions["score"] > THRESHOLD).astype("Int64").where(valid)
predictions["rank"] = predictions["score"].rank(ascending=False, method="first").astype("Int64")
predictions.to_csv(OUT_DIR / "predictions.csv", index=False)

print(f"\nSaved {len(predictions):,} rows to {OUT_DIR / 'predictions.csv'}")
display(predictions.sort_values("rank").head(10))

## 5. Metrics (if the file has labels)

In [ ]:
if not HAS_LABELS:
    print("No labels in the file; skipping metrics.")
else:
    scored = predictions[valid & predictions[LABEL_COL].notna()]
    metrics = compute_metrics(scored[LABEL_COL], scored["score"], hit_ks=HIT_KS, threshold=THRESHOLD)
    metrics = dict(n_molecules=len(scored), n_positives=int(scored[LABEL_COL].sum()), **metrics)
    (OUT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))
    display(pd.Series(metrics, name="value").to_frame().round(4))

In [ ]:
# Files produced
for path in sorted(OUT_DIR.rglob("*")):
    if path.is_file():
        print(f"{str(path.relative_to(OUT_DIR)):40s} {path.stat().st_size / 1e6:8.2f} MB")